In [1]:
import numpy as np
import zipfile
import re
import os
import copy
import random
import shutil
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from collections import defaultdict
from google.colab import files
from sklearn.metrics import classification_report, f1_score, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import pandas as pd

# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED             = 42

LOCAL_EPOCHS     = 30   # reduced to limit overfitting
FL_EPOCHS        = 15
PATIENCE         = 3    # strict early stopping
MIN_DELTA        = 0.02 # require meaningful improvement

D_MODEL          = 16   # compact model to reduce overfitting
N_HEADS          = 2
N_LAYERS         = 1
DROPOUT          = 0.5  # high dropout for regularization

LR               = 5e-4
FL_LR            = 1e-4
WEIGHT_DECAY     = 0.05 # strong L2 regularization

CLASS_NAMES      = ['Normal', 'Steady-DHSV', 'Transient-DHSV']
FL_ROUNDS        = 10
ALPHA            = 0.7
USE_GRAD_CLIP    = True

# full reproducibility across runs
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

print(f"Device        : {DEVICE}")
print(f"FL Rounds     : {FL_ROUNDS}")
print(f"LOCAL_EPOCHS  : {LOCAL_EPOCHS} (full local training)")
print(f"FL_EPOCHS     : {FL_EPOCHS} (fine-tuning per round)")
print(f"ALPHA         : {ALPHA} (stable aggregation)")
print(f"DROPOUT       : {DROPOUT} (regularization)")
print(f"WEIGHT_DECAY  : {WEIGHT_DECAY} (L2 regularization)")
print("\n" + "=" * 65)
print("🏆 GLOBAL MODEL — separate local training + FL rounds")
print("   • Phase 1: full local training per client")
print("   • Phase 2: FL rounds — fine-tuning only")
print("   • equal weight per client")
print("   • stable aggregation")
print("   • overfitting prevention: Dropout=0.5, WD=0.05")
print("=" * 65)

# ══════════════════════════════════════════════════════════════════════
# STEP 1: Upload ZIP
# ══════════════════════════════════════════════════════════════════════

print("\nUpload the preprocessed ZIP file\n")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

clean_name = re.sub(r'[^a-zA-Z0-9._-]', '_', file_name)
temp_zip = Path(f'/content/{clean_name}')
with open(temp_zip, 'wb') as f:
    f.write(uploaded[file_name])

extract_path = Path('/content/data_fl')
if extract_path.exists():
    shutil.rmtree(extract_path)
extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(temp_zip, 'r') as zf:
    zf.extractall(extract_path)
temp_zip.unlink()

print("ZIP contents:")
for f in sorted(extract_path.iterdir()):
    print(f"   {f.name} ({f.stat().st_size/1024:.1f} KB)")

# ══════════════════════════════════════════════════════════════════════
# STEP 2: Load Data
# ══════════════════════════════════════════════════════════════════════

clients_data    = {}
all_sensors_set = set()

for npz_path in sorted(extract_path.glob('*.npz')):
    client_id = npz_path.stem

    data    = np.load(npz_path, allow_pickle=True)
    X_train = data['X_train'].astype(np.float32)
    X_test  = data['X_test'].astype(np.float32)
    y_train = data['y_train'].astype(np.int64)
    y_test  = data['y_test'].astype(np.int64)

    sensors_csv = extract_path / f"{client_id}_sensors.csv"
    if sensors_csv.exists():
        sensors_df   = pd.read_csv(sensors_csv)
        sensor_names = sensors_df['sensor_name'].tolist()
    else:
        sensor_names = [f'Feature_{i}' for i in range(X_train.shape[2])]

    clients_data[client_id] = {
        'X_train'     : X_train,
        'X_test'      : X_test,
        'y_train'     : y_train,
        'y_test'      : y_test,
        'n_features'  : X_train.shape[2],
        'window_size' : X_train.shape[1],
        'sensor_names': sensor_names,
    }

    all_sensors_set.update(sensor_names)

    print(f"\n✅ {client_id}")
    print(f"   sensors={X_train.shape[2]} | train={X_train.shape[0]:,}")
    print(f"   sensor_names: {sensor_names}")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, cnt in zip(unique, counts):
        print(f"   {CLASS_NAMES[cls]}: {cnt:,} ({cnt/len(y_train)*100:.1f}%)")

ALL_SENSORS = sorted(list(all_sensors_set))
print(f"\n📡 All sensors: {ALL_SENSORS}")
print(f"   Total: {len(ALL_SENSORS)} sensors")
print(f"\ntotal clients: {len(clients_data)}")

# ══════════════════════════════════════════════════════════════════════
# MODEL
# ══════════════════════════════════════════════════════════════════════

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        n    = self.norm1(x)
        a, _ = self.attn(n, n, n)
        x    = x + a
        x    = x + self.ff(self.norm2(x))
        return x


class GlobalModel(nn.Module):
    def __init__(self, all_sensors, window_size=16, num_classes=3,
                 d_model=D_MODEL, n_heads=N_HEADS,
                 n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()

        self.all_sensors = all_sensors
        self.n_sensors   = len(all_sensors)
        self.d_model     = d_model

        self.sensor_embed = nn.Linear(window_size, d_model)

        self.sensor_pos = nn.ParameterDict({
            name.replace('-', '_'): nn.Parameter(
                torch.randn(d_model) * 0.02
            )
            for name in all_sensors
        })

        self.input_norm = nn.LayerNorm(d_model)
        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout)
            for _ in range(n_layers)
        ])

        self.out_norm   = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x, sensor_mask):
        x = x.permute(0, 2, 1)
        x = self.sensor_embed(x)

        pos = torch.stack([
            self.sensor_pos[name.replace('-', '_')]
            for name in self.all_sensors
        ], dim=0)
        x = x + pos.unsqueeze(0)

        if sensor_mask is not None:
            x = x * sensor_mask.unsqueeze(-1)

        x = self.input_norm(x)
        for blk in self.blocks:
            x = blk(x)

        if sensor_mask is not None:
            mask_sum = sensor_mask.sum(dim=1, keepdim=True) + 1e-8
            x = (x * sensor_mask.unsqueeze(-1)).sum(dim=1) / mask_sum
        else:
            x = x.mean(dim=1)

        x = self.dropout(self.out_norm(x))
        return self.classifier(x)

    def get_weights(self):
        return {name: param.data.clone()
                for name, param in self.named_parameters()}

    def load_weights(self, weights):
        state = self.state_dict()
        for k, v in weights.items():
            if k in state:
                state[k] = v.clone()
        self.load_state_dict(state)


# ══════════════════════════════════════════════════════════════════════
# SERVER
# ══════════════════════════════════════════════════════════════════════

def build_sensor_map(clients_data):
    sensor_to_clients = defaultdict(list)
    all_sensors       = set()

    for cid, data in clients_data.items():
        for sensor in data['sensor_names']:
            sensor_to_clients[sensor].append(cid)
            all_sensors.add(sensor)

    all_sensors    = sorted(list(all_sensors))
    shared_sensors = {}
    unique_sensors = {}

    for sensor, clients in sensor_to_clients.items():
        if len(clients) >= 2:
            shared_sensors[sensor] = clients
        else:
            unique_sensors[sensor] = clients[0]

    print("\n  🌐 Sensor Map (Auto-Discovered):")
    print("  " + "-" * 60)
    for sensor in all_sensors:
        clients = sensor_to_clients[sensor]
        n       = len(clients)
        if n >= 3:
            status = "✅ shared (all)"
        elif n >= 2:
            status = "🔄 shared (partial)"
        else:
            status = "🔒 unique"
        print(f"    {sensor:<25} → {n} client(s) {status}: {clients}")

    return {
        'sensor_to_clients': sensor_to_clients,
        'shared_sensors'   : shared_sensors,
        'unique_sensors'   : unique_sensors,
        'all_sensors'      : all_sensors,
    }


def fedavg_global_equal_stable(payloads, clients_data, global_weights, alpha=ALPHA):
    n_clients    = len(payloads)
    equal_weight = 1.0 / n_clients
    new_weights  = None

    for cid, weights in payloads.items():
        if new_weights is None:
            new_weights = {k: v.clone() * equal_weight
                          for k, v in weights.items()}
        else:
            for k in new_weights:
                if k in weights:
                    new_weights[k] += weights[k] * equal_weight

    if global_weights is not None:
        for k in new_weights:
            if k in global_weights:
                new_weights[k] = (alpha * new_weights[k] +
                                  (1 - alpha) * global_weights[k])

    print(f"\n  🌐 Server Aggregation (EQUAL WEIGHTS + Stable)")
    print(f"     Clients: {list(payloads.keys())}")
    print(f"     Each client weight: {equal_weight:.1%}")
    print(f"     Alpha (stability): {alpha}")

    total_samples = sum(len(clients_data[cid]['y_train']) for cid in payloads)
    for cid in payloads:
        size = len(clients_data[cid]['y_train'])
        pct  = size / total_samples * 100
        print(f"       {cid}: {size:,} ({pct:.1f}%) — equal weight")

    return new_weights


# ══════════════════════════════════════════════════════════════════════
# LOSS & UTILITIES
# ══════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets, class_weights=None):
        n      = logits.size(1)
        smooth = torch.zeros_like(logits).scatter_(
            1, targets.unsqueeze(1), 1.0)
        smooth = smooth * (1 - self.label_smoothing) + self.label_smoothing / n
        log_p  = F.log_softmax(logits, dim=1)
        ce     = -(smooth * log_p).sum(dim=1)
        pt     = torch.exp(-ce)
        loss   = (1 - pt) ** self.gamma * ce
        if class_weights is not None:
            loss = loss * class_weights[targets]
        return loss.mean()


class EarlyStopping:
    def __init__(self, patience=PATIENCE, min_delta=MIN_DELTA):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best      = None
        self.stop      = False

    def __call__(self, score):
        if self.best is None or score >= self.best + self.min_delta:
            self.best    = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop


def make_loaders(X_tr, y_tr, X_te, y_te, batch_size):
    Xtr = torch.tensor(X_tr, dtype=torch.float32)
    ytr = torch.tensor(y_tr, dtype=torch.long)
    Xte = torch.tensor(X_te, dtype=torch.float32)
    yte = torch.tensor(y_te, dtype=torch.long)

    uniq, cnts = torch.unique(ytr, return_counts=True)
    w_cls      = 1.0 / cnts.float()
    w_samp     = torch.zeros(len(ytr))
    for cls, w in zip(uniq, w_cls):
        w_samp[ytr == cls] = w
    sampler = WeightedRandomSampler(w_samp, len(ytr), replacement=True)

    class_w    = (w_cls / w_cls.sum()).to(DEVICE)

    tr_loader = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size,
        sampler=sampler, drop_last=True
    )
    tr_eval = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=False
    )
    te_loader = DataLoader(
        TensorDataset(Xte, yte), batch_size=batch_size, shuffle=False
    )
    return tr_loader, tr_eval, te_loader, class_w


# ══════════════════════════════════════════════════════════════════════
# EVALUATE
# ══════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate(model, loader, sensor_mask):
    model.eval()
    logits_all, labels_all = [], []
    for X, y in loader:
        mask = sensor_mask.expand(X.size(0), -1)
        logits_all.append(
            model(X.to(DEVICE), sensor_mask=mask).cpu())
        labels_all.append(y)
    logits = torch.cat(logits_all).numpy()
    labels = torch.cat(labels_all).numpy()
    preds  = np.argmax(logits, axis=1)
    return {
        'f1'      : f1_score(labels, preds, average='macro', zero_division=0),
        'accuracy': accuracy_score(labels, preds),
        'labels'  : labels,
        'preds'   : preds,
    }


def prepare_client_data(data, sensor_map):
    n_train    = len(data['X_train'])
    batch_size = 64 if n_train > 20000 else (32 if n_train > 5000 else 16)

    sensor_mask = torch.zeros(1, len(sensor_map['all_sensors']))
    for i, sensor in enumerate(sensor_map['all_sensors']):
        if sensor in data['sensor_names']:
            sensor_mask[0, i] = 1.0
    sensor_mask = sensor_mask.to(DEVICE)

    X_train_padded = torch.zeros(
        data['X_train'].shape[0],
        data['X_train'].shape[1],
        len(sensor_map['all_sensors']),
        dtype=torch.float32
    )
    X_test_padded = torch.zeros(
        data['X_test'].shape[0],
        data['X_test'].shape[1],
        len(sensor_map['all_sensors']),
        dtype=torch.float32
    )

    for j, sensor in enumerate(sensor_map['all_sensors']):
        if sensor in data['sensor_names']:
            idx = data['sensor_names'].index(sensor)
            X_train_padded[:, :, j] = torch.tensor(data['X_train'][:, :, idx])
            X_test_padded[:, :, j]  = torch.tensor(data['X_test'][:, :, idx])

    tr_loader, tr_eval, te_loader, class_w = make_loaders(
        X_train_padded.numpy(), data['y_train'],
        X_test_padded.numpy(),  data['y_test'],
        batch_size
    )

    return tr_loader, tr_eval, te_loader, class_w, sensor_mask


# ══════════════════════════════════════════════════════════════════════
# DETECT LARGEST CLIENT (for extended local training)
# ══════════════════════════════════════════════════════════════════════

def detect_largest_client(clients_data):
    max_size = 0
    largest_client = None

    for client_id in clients_data.keys():
        data_size = len(clients_data[client_id]['X_train'])
        if data_size > max_size:
            max_size = data_size
            largest_client = client_id

    return largest_client, max_size


# ══════════════════════════════════════════════════════════════════════
# DETECT WEAK CLIENT (for extra FL training)
# ══════════════════════════════════════════════════════════════════════

def detect_weak_client(clients_data, local_results):
    scores = {}

    max_size = max(len(clients_data[c]['X_train']) for c in clients_data)

    for client_id in clients_data.keys():
        data_size  = len(clients_data[client_id]['X_train'])
        client_f1  = local_results[client_id]['test']['f1']

        # smaller data → higher need; lower F1 → higher need
        normalized_size = 1.0 - (data_size / max_size if max_size > 0 else 0)
        normalized_f1   = 1.0 - client_f1

        need_score        = (normalized_size * 0.6) + (normalized_f1 * 0.4)
        scores[client_id] = need_score

    weak_client = max(scores, key=scores.get)
    weak_score  = scores[weak_client]

    if weak_score > 0.5:
        return weak_client, weak_score
    else:
        return None, weak_score


def train_model(model, tr_loader, tr_eval, te_loader,
                class_w, sensor_mask, epochs, lr):
    criterion  = FocalLoss(gamma=2.0, label_smoothing=0.1)
    optimizer  = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler  = ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    early_stop = EarlyStopping(patience=PATIENCE, min_delta=MIN_DELTA)

    best_f1, best_state, best_res = 0.0, None, None

    for epoch in range(1, epochs + 1):
        model.train()
        for X, y in tr_loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            mask   = sensor_mask.expand(X.size(0), -1)
            logits = model(X, sensor_mask=mask)
            loss   = criterion(logits, y, class_weights=class_w)
            loss.backward()
            if USE_GRAD_CLIP:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        train_res = evaluate(model, tr_eval, sensor_mask)
        test_res  = evaluate(model, te_loader, sensor_mask)
        scheduler.step(test_res['f1'])

        gap = train_res['f1'] - test_res['f1']
        if gap > 0.20 and epoch > 10:
            print(f"  ⚠️  Overfitting detected! Gap: {gap:.4f}")

        if test_res['f1'] > best_f1:
            best_f1    = test_res['f1']
            best_state = copy.deepcopy(model.state_dict())
            best_res   = {
                'train': train_res,
                'test' : test_res,
                'epoch': epoch
            }

        if early_stop(test_res['f1']):
            break

    model.load_state_dict(best_state)
    return best_res


# ══════════════════════════════════════════════════════════════════════
# BUILD SENSOR MAP
# ══════════════════════════════════════════════════════════════════════

sensor_map = build_sensor_map(clients_data)

# ══════════════════════════════════════════════════════════════════════
# LARGEST CLIENT DETECTION
# ══════════════════════════════════════════════════════════════════════

largest_client_id, largest_size = detect_largest_client(clients_data)

if largest_client_id is not None:
    print(f"\n🔍 Largest client detected:")
    print(f"   Client   : {largest_client_id}")
    print(f"   Data size: {largest_size:,} samples")
    print(f"   Extended local training: {int(LOCAL_EPOCHS * 1.5)} epochs")

# ══════════════════════════════════════════════════════════════════════
# PHASE 1: FULL LOCAL TRAINING
# ══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 65)
print("📚 Phase 1: Full Local Training")
print(f"   epochs: {LOCAL_EPOCHS} | lr: {LR}")
print("   each client trains independently — no weight sharing")
if largest_client_id is not None:
    print(f"   largest client ({largest_client_id}): {int(LOCAL_EPOCHS * 1.5)} epochs")
print("=" * 65)

local_results  = {}
client_loaders = {}

for client_id in sorted(clients_data.keys()):
    client_seed = SEED + sum(ord(c) for c in client_id)
    random.seed(client_seed)
    torch.manual_seed(client_seed)
    np.random.seed(client_seed)

    data = clients_data[client_id]

    print(f"\n  {'─'*60}")
    print(f"  [LOCAL] {client_id} | train={len(data['X_train']):,}")
    print(f"  sensors: {data['sensor_names']}")

    model = GlobalModel(
        all_sensors=sensor_map['all_sensors'],
        window_size=data['window_size'],
        d_model=D_MODEL, n_heads=N_HEADS,
        n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)

    tr_loader, tr_eval, te_loader, class_w, sensor_mask = \
        prepare_client_data(data, sensor_map)

    print(f"  📊 Mask: {sensor_mask.squeeze().tolist()}")

    current_local_epochs = LOCAL_EPOCHS
    if client_id == largest_client_id:
        current_local_epochs = int(LOCAL_EPOCHS * 1.5)
        print(f"  extended local training: {current_local_epochs} epochs (largest dataset)")

    res = train_model(model, tr_loader, tr_eval, te_loader,
                      class_w, sensor_mask, current_local_epochs, LR)

    local_results[client_id]  = res
    client_loaders[client_id] = (tr_loader, tr_eval, te_loader,
                                  class_w, sensor_mask, model)

    torch.cuda.empty_cache()

    print(f"  ✅ best epoch: {res['epoch']} | "
          f"train F1: {res['train']['f1']:.4f} | "
          f"test F1:  {res['test']['f1']:.4f}")

    gap = res['train']['f1'] - res['test']['f1']
    if gap > 0.15:
        print(f"  ⚠️  Overfitting Gap: {gap:.4f} (Train-Test)")

avg_local_f1  = np.mean([r['test']['f1']      for r in local_results.values()])
avg_local_acc = np.mean([r['test']['accuracy'] for r in local_results.values()])

print(f"\n{'─'*45}")
print(f"📊 Local Baseline Summary (before FL):")
print(f"{'Client':<12} {'Test F1':>10} {'Accuracy':>10} {'Overfit Gap':>12}")
print(f"{'─'*50}")
for cid, res in sorted(local_results.items()):
    gap = res['train']['f1'] - res['test']['f1']
    print(f"{cid:<12} {res['test']['f1']:>10.4f} "
          f"{res['test']['accuracy']:>10.4f} {gap:>12.4f}")
print(f"{'─'*50}")
print(f"{'Local Avg':<12} {avg_local_f1:>10.4f} {avg_local_acc:>10.4f}")

print(f"\nLocal training complete — starting FL rounds")

# ══════════════════════════════════════════════════════════════════════
# WEAK CLIENT DETECTION
# ══════════════════════════════════════════════════════════════════════

weak_client_id, weak_score = detect_weak_client(clients_data, local_results)

if weak_client_id is not None:
    print(f"\n🔍 Weak client detected (needs extra FL training):")
    print(f"   Client    : {weak_client_id}")
    print(f"   Need score: {weak_score:.3f}")
    print(f"   Will train with 30 epochs per FL round (default: {FL_EPOCHS})")
else:
    print(f"\nAll clients are balanced — no extra FL training needed")

# ══════════════════════════════════════════════════════════════════════
# PHASE 2: FL ROUNDS
# ══════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 65)
print("🚀 Phase 2: Federated Learning Rounds")
print(f"   clients      : {list(clients_data.keys())}")
print(f"   FL rounds    : {FL_ROUNDS}")
print(f"   FL epochs    : {FL_EPOCHS} fine-tuning per round")
if weak_client_id is not None:
    print(f"   weak client ({weak_client_id}): 30 epochs")
print(f"   FL lr        : {FL_LR} (lower than local)")
print(f"   Total sensors: {len(ALL_SENSORS)}")
print(f"   Stable Agg   : α={ALPHA}")
print(f"   Equal Weights: ✅")
print(f"   Class Weights: ✅ WeightedRandomSampler")
print(f"   Overfitting Prevention: Dropout=0.5, WD=0.05, LS=0.1")
print("=" * 65)

global_weights      = None
fl_history          = []
best_global_f1      = avg_local_f1
best_global_results = copy.deepcopy(local_results)
best_round          = 0

for fl_round in range(1, FL_ROUNDS + 1):
    print(f"\n{'='*65}")
    print(f"  🌐 FL Round {fl_round}/{FL_ROUNDS}")
    print(f"{'='*65}")

    round_payloads = {}
    round_results  = {}

    for client_id in sorted(clients_data.keys()):
        client_seed = SEED + sum(ord(c) for c in client_id) + fl_round
        random.seed(client_seed)
        torch.manual_seed(client_seed)
        np.random.seed(client_seed)

        data = clients_data[client_id]
        tr_loader, tr_eval, te_loader, class_w, sensor_mask, model = \
            client_loaders[client_id]

        current_epochs = FL_EPOCHS
        is_weak = False

        if weak_client_id is not None and client_id == weak_client_id:
            current_epochs = 30
            is_weak = True

        print(f"\n  {'─'*60}")
        print(f"  [FL] {client_id} | round={fl_round} | "
              f"epochs={current_epochs} | lr={FL_LR}")
        if is_weak:
            print(f"  extended training for weak client: {current_epochs} epochs")

        if global_weights is not None:
            model.load_weights(global_weights)
            print(f"  ✅ loaded global model")

        res = train_model(model, tr_loader, tr_eval, te_loader,
                          class_w, sensor_mask, current_epochs, FL_LR)

        client_loaders[client_id] = (tr_loader, tr_eval, te_loader,
                                      class_w, sensor_mask, model)

        round_payloads[client_id] = model.get_weights()
        round_results[client_id]  = res

        torch.cuda.empty_cache()

        print(f"  ✅ best epoch: {res['epoch']} | "
              f"train F1: {res['train']['f1']:.4f} | "
              f"test F1:  {res['test']['f1']:.4f}")

    print(f"\n  🌐 Server aggregating...")
    global_weights = fedavg_global_equal_stable(
        round_payloads, clients_data, global_weights, alpha=ALPHA
    )

    avg_f1  = np.mean([r['test']['f1']      for r in round_results.values()])
    avg_acc = np.mean([r['test']['accuracy'] for r in round_results.values()])
    delta   = avg_f1 - avg_local_f1
    sign    = '+' if delta >= 0 else ''

    if avg_f1 > best_global_f1:
        best_global_f1      = avg_f1
        best_global_results = copy.deepcopy(round_results)
        best_round          = fl_round
        print(f"  ⭐️ New best! avg F1={avg_f1:.4f} "
              f"({sign}{delta:.4f} vs Local Baseline)")

    fl_history.append({
        'round'  : fl_round,
        'avg_f1' : avg_f1,
        'avg_acc': avg_acc,
        'results': copy.deepcopy(round_results),
    })

    print(f"\n  Round {fl_round} Summary:")
    print(f"  {'Client':<12} {'Test F1':>10} {'Accuracy':>10} {'vs Local':>10}")
    print(f"  {'─'*50}")
    for cid, res in sorted(round_results.items()):
        d = res['test']['f1'] - local_results[cid]['test']['f1']
        s = '+' if d >= 0 else ''
        print(f"  {cid:<12} {res['test']['f1']:>10.4f} "
              f"{res['test']['accuracy']:>10.4f} {s}{d:>9.4f}")
    print(f"  {'─'*50}")
    print(f"  {'Global Avg':<12} {avg_f1:>10.4f} {avg_acc:>10.4f} "
          f"{sign}{delta:>9.4f}")


# ══════════════════════════════════════════════════════════════════════
# FINAL RESULTS
# ══════════════════════════════════════════════════════════════════════

print("\n\n" + "=" * 65)
if best_round == 0:
    print("🏁 FINAL RESULTS — Local Baseline (FL did not surpass local)")
else:
    print(f"🏁 FINAL RESULTS — Best Round: {best_round} "
          f"(avg F1={best_global_f1:.4f})")
print("=" * 65)

last = best_global_results

print(f"\n{'Client':<12} {'Sensors':>8} {'Local F1':>10} "
      f"{'FL F1':>10} {'Delta':>8} {'Accuracy':>10} {'Overfit Gap':>12}")
print("─" * 75)
for cid, res in sorted(last.items()):
    d        = clients_data[cid]
    local_f1 = local_results[cid]['test']['f1']
    fl_f1    = res['test']['f1']
    delta    = fl_f1 - local_f1
    sign     = '+' if delta >= 0 else ''
    gap      = res['train']['f1'] - res['test']['f1']
    flag     = '✅' if fl_f1 >= 0.80 else ('⚠️' if fl_f1 >= 0.65 else '❌')
    print(f"{cid:<12} {d['n_features']:>8} {local_f1:>10.4f} "
          f"{fl_f1:>10.4f} {sign}{delta:>7.4f} "
          f"{res['test']['accuracy']:>10.4f} {gap:>12.4f}  {flag}")

avg_fl_f1   = np.mean([r['test']['f1']      for r in last.values()])
avg_fl_acc  = np.mean([r['test']['accuracy'] for r in last.values()])
delta_final = avg_fl_f1 - avg_local_f1
sign        = '+' if delta_final >= 0 else ''
print("─" * 75)
print(f"{'Global Avg':<12} {'':>8} {avg_local_f1:>10.4f} "
      f"{avg_fl_f1:>10.4f} {sign}{delta_final:>7.4f} {avg_fl_acc:>10.4f}")

print("\n" + "=" * 65)
print("📈 F1 Progress — Local Baseline vs FL Rounds")
print("=" * 65)
print(f"{'Round':>6} ", end="")
for cid in sorted(clients_data.keys()):
    print(f"{cid:>14}", end="")
print(f"{'Avg F1':>10}")
print("─" * 65)

print(f"{'Local':>6} ", end="")
for cid in sorted(clients_data.keys()):
    print(f"{local_results[cid]['test']['f1']:>14.4f}", end="")
print(f"{avg_local_f1:>10.4f}  ← baseline")

for h in fl_history:
    marker = " ⭐️" if h['round'] == best_round else ""
    print(f"{h['round']:>6} ", end="")
    for cid in sorted(clients_data.keys()):
        print(f"{h['results'][cid]['test']['f1']:>14.4f}", end="")
    print(f"{h['avg_f1']:>10.4f}{marker}")

print("\n" + "=" * 65)
print("Classification Reports — Best Results")
print("=" * 65)
for cid, res in sorted(last.items()):
    d = clients_data[cid]
    print(f"\n{cid} | sensors: {d['sensor_names']}")
    print(classification_report(
        res['test']['labels'], res['test']['preds'],
        target_names=CLASS_NAMES, zero_division=0
    ))

print(f"\n✅ Done!")
print(f"   Local Baseline avg F1 : {avg_local_f1:.4f}")
print(f"   Best FL avg F1        : {best_global_f1:.4f}")
delta = best_global_f1 - avg_local_f1
sign  = '+' if delta >= 0 else ''
print(f"   FL Improvement        : {sign}{delta:.4f}")
if best_round == 0:
    print(f"   FL did not surpass local baseline")
else:
    print(f"   FL surpassed local baseline at round {best_round}")

print("\n" + "=" * 65)
print("🛡  Overfitting Prevention Techniques:")
print(f"   Dropout       : {DROPOUT}")
print(f"   Weight Decay  : {WEIGHT_DECAY}")
print(f"   Label Smoothing: 0.1")
print(f"   Model size    : D_MODEL={D_MODEL}")
print(f"   Local epochs  : {LOCAL_EPOCHS}")
print(f"   Early Stopping: patience={PATIENCE}, min_delta={MIN_DELTA}")
if largest_client_id is not None:
    print(f"   Extended local training for largest client: {largest_client_id}")
if weak_client_id is not None:
    print(f"   Extended FL training for weak client: 30 epochs")
print("=" * 65)

Device        : cpu
FL Rounds     : 10
LOCAL_EPOCHS  : 30 (full local training)
FL_EPOCHS     : 15 (fine-tuning per round)
ALPHA         : 0.7 (stable aggregation)
DROPOUT       : 0.5 (regularization)
WEIGHT_DECAY  : 0.05 (L2 regularization)

🏆 GLOBAL MODEL — separate local training + FL rounds
   • Phase 1: full local training per client
   • Phase 2: FL rounds — fine-tuning only
   • equal weight per client
   • stable aggregation
   • overfitting prevention: Dropout=0.5, WD=0.05

Upload the preprocessed ZIP file



Saving wells_4clients_with_sensors (1)_without_client13.zip to wells_4clients_with_sensors (1)_without_client13.zip
ZIP contents:
   client_1.npz (470.9 KB)
   client_1_README.txt (0.7 KB)
   client_1_sensors.csv (0.1 KB)
   client_2.npz (1925.3 KB)
   client_2_README.txt (0.7 KB)
   client_2_sensors.csv (0.1 KB)
   client_3.npz (454.9 KB)
   client_3_README.txt (0.7 KB)
   client_3_sensors.csv (0.1 KB)
   common_sensors.csv (0.0 KB)

✅ client_1
   sensors=5 | train=10,527
   sensor_names: ['P-ANULAR', 'P-MON-CKP', 'P-PDG', 'P-TPT', 'T-TPT']
   Normal: 4,786 (45.5%)
   Steady-DHSV: 1,961 (18.6%)
   Transient-DHSV: 3,780 (35.9%)

✅ client_2
   sensors=5 | train=55,803
   sensor_names: ['P-ANULAR', 'P-PDG', 'P-TPT', 'T-PDG', 'T-TPT']
   Normal: 16,499 (29.6%)
   Steady-DHSV: 8,371 (15.0%)
   Transient-DHSV: 30,933 (55.4%)

✅ client_3
   sensors=8 | train=7,366
   sensor_names: ['ABER-CKP', 'ESTADO-PXO', 'P-ANULAR', 'P-MON-CKP', 'P-PDG', 'P-TPT', 'T-PDG', 'T-TPT']
   Normal: 5,107 (69.3%)